Take a copy of the folder stuctures first before acting on the Data Archive.

In [ ]:
Perfect — then we’ll add a **cluster summary step** before writing categories back into the notes. That way you can see which tags dominate in each cluster and assign human-readable names (instead of `Cluster-0`, etc.).

Here’s the extended version:

#clustering #ml_process #preprocessing #data_management #code_snippet

```python
import os
import yaml
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter

def load_notes(folder_path):
    notes = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".md"):
                filepath = os.path.join(root, file)
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                if content.startswith("---"):
                    parts = content.split("---")
                    if len(parts) > 2:
                        yaml_block = yaml.safe_load(parts[1])
                        tags = yaml_block.get("tags", [])
                        notes.append({
                            "path": filepath,
                            "tags": tags,
                            "yaml": yaml_block,
                            "content": parts
                        })
    return notes

def cluster_notes(notes, n_clusters=5):
    all_tags = [note["tags"] for note in notes]
    mlb = MultiLabelBinarizer()
    tag_matrix = mlb.fit_transform(all_tags)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(tag_matrix)

    for note, label in zip(notes, labels):
        note["cluster"] = f"Cluster-{label}"
    return notes, labels, mlb.classes_

def summarize_clusters(notes, labels, tag_names):
    cluster_summary = {}
    for label in set(labels):
        cluster_tags = []
        for note, cluster in zip(notes, labels):
            if cluster == label:
                cluster_tags.extend(note["tags"])
        common_tags = Counter(cluster_tags).most_common(10)
        cluster_summary[label] = common_tags

    print("\n=== Cluster Summaries ===")
    for label, common_tags in cluster_summary.items():
        print(f"\nCluster-{label}:")
        for tag, count in common_tags:
            print(f"  {tag}: {count}")
    return cluster_summary

def update_notes(notes, cluster_rename=None):
    """
    cluster_rename: dict mapping cluster name -> new category label
    """
    for note in notes:
        yaml_block = note["yaml"]
        cluster_label = note["cluster"]
        category = cluster_rename.get(cluster_label, cluster_label) if cluster_rename else cluster_label

        if "category" not in yaml_block or not yaml_block["category"]:
            yaml_block["category"] = category
            note["content"][1] = yaml.dump(yaml_block, sort_keys=False)
            new_content = "---\n" + note["content"][1] + "---".join(note["content"][2:])
            with open(note["path"], "w", encoding="utf-8") as f:
                f.write(new_content)
            print(f"Updated {note['path']} -> {category}")

# Example usage:
# notes = load_notes("/path/to/your/obsidian/notes")
# clustered_notes, labels, tag_names = cluster_notes(notes, n_clusters=5)
# cluster_summary = summarize_clusters(clustered_notes, labels, tag_names)
# 
# # Optionally rename clusters based on summaries
# rename_map = {
#     "Cluster-0": "Data Engineering",
#     "Cluster-1": "Machine Learning",
#     "Cluster-2": "Maths",
#     # etc.
# }
# update_notes(clustered_notes, cluster_rename=rename_map)
```

---

### Workflow with this script

1. Run clustering.
2. Inspect printed **cluster summaries** (top tags per cluster).
3. Build a mapping (e.g. `Cluster-0 → Data Engineering`).
4. Run `update_notes` with your mapping.

